In [ ]:
!pip install qiskit
!pip install qiskit-ibm-runtime
!pip install qiskit_aer
!pip install pylatexenc

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler

In [ ]:
def build_oracle(strings_one):
    # If the function is never 1, the oracle is the identity.
    # Hence, we return an empty circuit.
    if len(strings_one) == 0:
        return QuantumCircuit()

    # Number of bits that the function takes as input:
    n = len(strings_one[0])

    qc = QuantumCircuit(n+1)
    for x in strings_one:
        # Find the positions in the string x where the bit is 0.
        # For this, we find the list of indices i such that x[i]=='0'.
        bits_zero = []
        for i in range(len(x)):
            val = x[i]
            if val == '0':
                bits_zero.append(i)

        # Step 1 in our construction.
        for bit in bits_zero:
            qc.x(bit)

        # Step 2.
        qc.mcx(list(range(n)), n)

        # Step 3.
        for bit in bits_zero:
            qc.x(bit)

    return qc

In [ ]:
build_oracle(["000", "100", "111"]).draw("mpl")

In [ ]:
def DJ(oracle):
    # Number of qubits in the circuit (same as the oracle).
    # If we are working with an n-bit function, nqubits = n + 1.
    nqubits = oracle.num_qubits

    # Create a circuit with (nqubits) qubits and (nqubits-1) bits.
    # Remember we are only going to measure the first (nqubits-1) qubits!
    qc = QuantumCircuit(nqubits, nqubits - 1)

    # Initialize the bottom qubit to |1>.
    qc.x(nqubits - 1)

    for i in range(nqubits):
        qc.h(i)
    qc.barrier()
    qc = qc.compose(oracle)
    qc.barrier()
    for i in range(nqubits - 1):
        qc.h(i)
        qc.measure(i,i)
    return qc

In [ ]:
oracle = QuantumCircuit(4)
dj = DJ(oracle)
dj.draw("mpl")

In [ ]:
backend = AerSimulator(seed_simulator = 18620123)
sampler = Sampler(backend)

job = sampler.run([dj], shots = 1)
result = job.result()[0].data.c
print(result.get_bitstrings())

In [ ]:
# Oracle of a balanced function (takes the value 1 with four inputs):
oracle = build_oracle(["000", "101", "100", "110"])
# Prepare the D-J circuit:
dj = DJ(oracle)
# Run it!
job = sampler.run([dj], shots = 1)
result = job.result()[0].data.c
print(result.get_bitstrings())